In [1]:
from data import X_train, X_test, y_train, y_test
import torch
import plotly.express as px
import numpy as np



In [3]:
a = X_train[0]
torch.tensor(a.toarray(), dtype=torch.float)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is available")
elif torch.cuda.mps.is_available():
    device = torch.device("cuda")
    print("cuda is available")
else:
    device = torch.device("cuda")
    print("cuda and mps not detected")
print(device)


MPS is available
mps


In [4]:
print(torch.cuda.is_available())

False


In [5]:
import torch
import torch.nn as nn

class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc = nn.TransformerEncoderLayer(
            d_model=20000,
            nhead=16,
            batch_first=True
        )

        self.transformer_enc = nn.TransformerEncoder(
            encoder_layer=self.enc,
            num_layers=2
        )

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(20000, 1)
        )

    def forward(self, x):
        # x should be shape: (batch_size, 20000)

        x = x.unsqueeze(1)
        # shape: (batch_size, 1, 20000)

        x = self.transformer_enc(x)
        # shape: (batch_size, 1, 20000)

        x = x.mean(dim=1)
        # shape: (batch_size, 20000)

        x = self.classifier(x)
        # shape: (batch_size, 1)

        return x

model = TransformerModel().to(device)
a = torch.tensor(X_train[0].toarray(), dtype=torch.float).to(device)


In [6]:
a.shape

torch.Size([1, 20000])

In [9]:
model(a)

tensor([[-0.1719]], device='mps:0', grad_fn=<LinearBackward0>)

In [ ]:
from torch.optim import Adam
from tqdm import tqdm
optimizer = Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()
model = model.to(device)
num_epochs = 5

for epoch in tqdm(range(num_epochs)):
    for x, y in zip(X_train, y_train):
        x = torch.tensor(x.toarray(), dtype=torch.float).to(device)
        y = torch.tensor(y.toarray(), dtype=torch.float).to(device)

        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


